# Project 5 -- Vinay Gannamaneni

**TA Help:** (for instance) John Smith, Alice Jones, etc., list names of any TAs who helped you

- For example: Help with figuring out how to write a function (describe the tasks that they helped you with)

**Collaboration:** My Friend in CS, My Uncle, Another Student, etc., list names of any other people who helped you

(describe the tasks that they helped you with)
- For example: helped figuring out how to load the dataset.
- Another example: helped debug error with my plot.

**Internet Resources:** Stack Exchange, Stack Overflow, etc.

(describe any information that you learned from internet resources, including the URLs)
- data frames in Pandas versus R from StackOverflow  https://stackoverflow.com/questions/8991709/why-were-pandas-merges-in-python-faster-than-data-table-merges-in-r-in-2012

**ChatGPT, Gemini, Claude, etc:** Any language models or generative AI chatbots that helped you.

(if you used any such tools, please tell us here)
- For example:  I asked ChatGPT how to define a new data frames
- Another example:  Gemini told me how to make a function for sorting my data

- ***Link to AI Chat History***: Please share a link to your chat if you used AI (ex. ChatGPT Shared Links)
**OVERALL MESSAGE:** Any time that you used anything except your brain to solve the questions in these projects, you need to disclose such resources at the start of the project, with details about your usage of the tools.

**YOUR OWN WORK:** Even when you utilize other resources, do NOT just copy and paste.  Write all explanations in your own words, using several sentences in English, which are understandable and which you wrote (and did not just copy and paste).

## Benchmarking pandas vs polars read speed on flights and baseball data

In [3]:
import pandas as pd
import polars as pl
import time

In [4]:
start = time.time()
pandas_df_1987 = pd.read_csv("/anvil/projects/tdm/data/flights/subset/1987.csv")
end = time.time()
print(f"Pandas read_csv took {end - start} seconds")

Pandas read_csv took 1.620758056640625 seconds


``time.time()`` captures the current time before and after reading.

In [5]:
start = time.time()
polars_df_1987 = pl.read_csv("/anvil/projects/tdm/data/flights/subset/1987.csv", null_values=["NA"])
end = time.time()
print(f"Polars read_csv took {end - start} seconds")

Polars read_csv took 0.8763480186462402 seconds


``null_values=["NA"]`` tells Polars to treat "NA" as missing values.

In [6]:
start = time.time()
pandas_df = pd.read_csv('/anvil/projects/tdm/data/lahman/data/AllstarFull.csv', header=None)
pandas_df.columns = ['playerID', 'yearID', 'gameNum', 'gameID', 'teamID', 'lgID', 'GP', 'startingPos']
end = time.time()
print(f"Pandas read_csv took {end - start} seconds")

Pandas read_csv took 0.008926630020141602 seconds


``header=None`` tells pandas there's no header row.

In [7]:
start = time.time()
polars_df = pl.read_csv('/anvil/projects/tdm/data/lahman/data/AllstarFull.csv', has_header=False)
polars_df.columns = ['playerID', 'yearID', 'gameNum', 'gameID', 'teamID', 'lgID', 'GP', 'startingPos']
end = time.time()
print(f"Polars read_csv took {end - start} seconds")

Polars read_csv took 0.007295846939086914 seconds


``has_header=False`` is Polars' way of saying no header row.

In [8]:
pandas_df.head()

,playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
0,kalinal01,1957,0,NLS195707090,DET,AL,1,9.0
1,demaejo01,1957,0,NLS195707090,KC1,AL,0,NaN
2,grimbo01,1957,0,NLS195707090,NYA,AL,1,NaN
3,howarel01,1957,0,NLS195707090,NYA,AL,0,NaN
4,loesbi01,1957,0,NLS195707090,BAL,AL,1,NaN


In [9]:
polars_df.head()

playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
str,i64,i64,str,str,str,i64,i64
"""kalinal01""",1957,0,"""NLS195707090""","""DET""","""AL""",1,9
"""demaejo01""",1957,0,"""NLS195707090""","""KC1""","""AL""",0,null
"""grimbo01""",1957,0,"""NLS195707090""","""NYA""","""AL""",1,null
"""howarel01""",1957,0,"""NLS195707090""","""NYA""","""AL""",0,null
"""loesbi01""",1957,0,"""NLS195707090""","""BAL""","""AL""",1,null


Some differences noticed:
- Polars displays data types more clearly and explicitly in the output
- Polars formatting is different from Pandas (cleaner table structure)
- Polars is generally faster at reading data, especially larger files
- Syntax differs: ``header=None`` (Pandas) vs ``has_header=False`` (Polars)

## Selecting and filtering columns in both libraries

In [10]:
pandas_df[['playerID', 'teamID']]

,playerID,teamID
0,kalinal01,DET
1,demaejo01,KC1
2,grimbo01,NYA
3,howarel01,NYA
4,loesbi01,BAL
...,...,...
5668,mantijo01,ARI
5669,mikolmi01,SLN
5670,musgrjo01,SDN
5671,rodonca01,SFN


Pandas uses bracket notation with a list of column names.

In [11]:
polars_df.select(['playerID', 'teamID'])

playerID,teamID
str,str
"""kalinal01""","""DET"""
"""demaejo01""","""KC1"""
"""grimbo01""","""NYA"""
"""howarel01""","""NYA"""
"""loesbi01""","""BAL"""
…,…
"""mantijo01""","""ARI"""
"""mikolmi01""","""SLN"""
"""musgrjo01""","""SDN"""


Polars uses ``.select()`` method with a list of column names.

In [12]:
philly_players_pandas = pandas_df[pandas_df['teamID'] == "PHI"]
philly_players_pandas.head()

,playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
17,simmocu01,1957,0,NLS195707090,PHI,NL,1,1.0
38,sanfoja02,1957,0,NLS195707090,PHI,NL,1,NaN
77,ashburi01,1958,0,ALS195807080,PHI,NL,0,NaN
80,farretu01,1958,0,ALS195807080,PHI,NL,1,NaN
176,conlege01,1959,1,NLS195907070,PHI,NL,0,NaN


Pandas uses bracket notation with a boolean condition inside and this shows the first 5 Philadelphia players from the Pandas dataframe.

In [13]:
philly_players_polars = polars_df.filter(pl.col("teamID") == "PHI")
philly_players_polars.head()

playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
str,i64,i64,str,str,str,i64,i64
"""simmocu01""",1957,0,"""NLS195707090""","""PHI""","""NL""",1,1
"""sanfoja02""",1957,0,"""NLS195707090""","""PHI""","""NL""",1,null
"""ashburi01""",1958,0,"""ALS195807080""","""PHI""","""NL""",0,null
"""farretu01""",1958,0,"""ALS195807080""","""PHI""","""NL""",1,null
"""conlege01""",1959,1,"""NLS195907070""","""PHI""","""NL""",0,null


Polars uses ``.filter()`` method with ``pl.col()`` to specify the column and this shows the first 5 Philadelphia players from the Polars dataframe.

In [14]:
philly_players_pandas.sort_values('yearID')

,playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
2334,bartedi01,1933,0,ALS193307060,PHI,NL,1,6.0
2337,kleinch01,1933,0,ALS193307060,PHI,NL,1,9.0
2411,wilsoji01,1935,0,ALS193507080,PHI,NL,1,2.0
2457,whitnpi01,1936,0,NLS193607070,PHI,NL,1,5.0
2472,waltebu01,1937,0,ALS193707070,PHI,NL,1,NaN
...,...,...,...,...,...,...,...,...
5512,schwaky01,2022,0,NLS202207190,PHI,NL,1,NaN
5655,harpebr03,2022,0,NLS202207190,PHI,NL,0,NaN
5654,schwaky01,2022,0,NLS202207190,PHI,NL,1,NaN
5548,kimbrcr01,2023,0,ALS202307110,PHI,NL,1,NaN


Pandas uses ``.sort_values()`` to sort by a column.

In [15]:
philly_players_polars.sort('yearID')

playerID,yearID,gameNum,gameID,teamID,lgID,GP,startingPos
str,i64,i64,str,str,str,i64,i64
"""bartedi01""",1933,0,"""ALS193307060""","""PHI""","""NL""",1,6
"""kleinch01""",1933,0,"""ALS193307060""","""PHI""","""NL""",1,9
"""wilsoji01""",1935,0,"""ALS193507080""","""PHI""","""NL""",1,2
"""whitnpi01""",1936,0,"""NLS193607070""","""PHI""","""NL""",1,5
"""waltebu01""",1937,0,"""ALS193707070""","""PHI""","""NL""",1,null
…,…,…,…,…,…,…,…
"""schwaky01""",2022,0,"""NLS202207190""","""PHI""","""NL""",1,null
"""schwaky01""",2022,0,"""NLS202207190""","""PHI""","""NL""",1,null
"""harpebr03""",2022,0,"""NLS202207190""","""PHI""","""NL""",0,null


Polars uses ``.sort()`` to sort by a column.

## Filtering Indianapolis flights and selecting scheduled time columns with regex

Patterns noticed:
- Polars uses ``.filter()`` while Pandas uses bracket notation for filtering
- Polars requires ``pl.col()`` to explicitly specify which column to operate on
- Polars uses ``.sort()`` while Pandas uses ``.sort_values()``
- Polars uses ``.select()`` while Pandas uses bracket notation for selecting columns
- Polars is more explicit and structured and it clearly states what operation is happening
- Polars syntax is more verbose but clearer about intent

In [16]:
myDF = pl.read_csv("/anvil/projects/tdm/data/flights/subset/2005.csv")

This dataset is too big for 2 cores.

In [17]:
indy_flights = myDF.filter(pl.col("Origin") == "IND")

This creates a new dataframe with only flights originating from Indianapolis.

In [18]:
indy_flights.select(pl.col("^CRS.*$"))

CRSDepTime,CRSArrTime,CRSElapsedTime
i64,i64,i64
1602,1614,72
1602,1614,72
1602,1614,72
1602,1614,72
1602,1614,72
…,…,…
730,912,102
1621,1754,93
1200,1330,90


The regex pattern ``"^CRS.*$"`` means:
- ``^`` = start of string
- ``CRS`` = must begin with these exact characters
- ``.*`` = followed by any characters (or nothing)
- ``$`` = end of string

## Selecting delay-related columns with regex patterns

In [19]:
indy_flights.select(pl.col("^.*Delay.*$"))

ArrDelay,DepDelay,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
str,str,i64,i64,i64,i64,i64
"""37""","""39""",8,0,0,0,29
"""10""","""25""",0,0,0,0,0
"""83""","""84""",0,0,83,0,0
"""-11""","""5""",0,0,0,0,0
"""194""","""192""",0,0,130,0,64
…,…,…,…,…,…,…
"""-10""","""2""",0,0,0,0,0
"""3""","""-2""",0,0,0,0,0
"""-12""","""-3""",0,0,0,0,0


The pattern ``"^.*Delay.*$"`` means:
- ``^`` = start of string
- ``.*`` = any characters (or nothing) before
- ``Delay`` = must contain this word
- ``.*`` = any characters (or nothing) after
- ``$`` = end of string

In [20]:
indy_flights.select(pl.col("^.*Delay.*$")).columns

['ArrDelay',
 'DepDelay',
 'CarrierDelay',
 'WeatherDelay',
 'NASDelay',
 'SecurityDelay',
 'LateAircraftDelay']

In [21]:
indy_flights.select(pl.col("^(Arr|Dep).*$"))

DepTime,ArrTime,ArrDelay,DepDelay
str,str,str,str
"""1641""","""1651""","""37""","""39"""
"""1627""","""1624""","""10""","""25"""
"""1726""","""1737""","""83""","""84"""
"""1607""","""1603""","""-11""","""5"""
"""1914""","""1928""","""194""","""192"""
…,…,…,…
"""732""","""902""","""-10""","""2"""
"""1619""","""1757""","""3""","""-2"""
"""1157""","""1318""","""-12""","""-3"""


The pattern ``"^(Arr|Dep).*$"`` means:
- ``^`` = start of string
- ``(Arr|Dep)`` = must start with EITHER 'Arr' OR 'Dep'
- ``.*`` = followed by any characters (or nothing)
- ``$`` = end of string

In [22]:
indy_flights.select(pl.col("^(Arr|Dep).*$")).head()

DepTime,ArrTime,ArrDelay,DepDelay
str,str,str,str
"""1641""","""1651""","""37""","""39"""
"""1627""","""1624""","""10""","""25"""
"""1726""","""1737""","""83""","""84"""
"""1607""","""1603""","""-11""","""5"""
"""1914""","""1928""","""194""","""192"""


In [23]:
indy_flights.select(pl.col("^(Arr|Dep).*Delay$"))

ArrDelay,DepDelay
str,str
"""37""","""39"""
"""10""","""25"""
"""83""","""84"""
"""-11""","""5"""
"""194""","""192"""
…,…
"""-10""","""2"""
"""3""","""-2"""
"""-12""","""-3"""


The pattern ``"^(Arr|Dep).*Delay$"`` means:
- ``^`` = start of string
- ``(Arr|Dep)`` = must start with 'Arr' or 'Dep'
- ``.*`` = any characters in between (or nothing)
- ``Delay`` = must end with 'Delay'

## Casting arrival time and computing the absolute scheduling difference

In [25]:
myDF = myDF.with_columns(
    pl.col("ArrTime").cast(pl.Int64, strict=False)
)

ArrTime contains military time like "1830" which needs to be numeric for calculations.

In [26]:
myDF = myDF.with_columns(
    (pl.col("ArrTime") - pl.col("CRSArrTime")).abs().alias("AbsDiffArrTime")
)

This creates a new column called AbsDiffArrTime and shows the actual arrival time differs from scheduled in absolute terms.

In [27]:
myDF.head()

Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,AbsDiffArrTime
i64,i64,i64,i64,str,i64,i64,i64,str,i64,str,str,i64,str,str,str,str,str,i64,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64
2005,1,28,5,"""1603""",1605,1741,1759,"""UA""",541,"""N935UA""","""158""",174,"""131""","""-18""","""-2""","""BOS""","""ORD""",867,4,23,0,null,0,0,0,0,0,0,18
2005,1,29,6,"""1559""",1605,1736,1759,"""UA""",541,"""N941UA""","""157""",174,"""136""","""-23""","""-6""","""BOS""","""ORD""",867,6,15,0,null,0,0,0,0,0,0,23
2005,1,30,7,"""1603""",1610,1741,1805,"""UA""",541,"""N342UA""","""158""",175,"""131""","""-24""","""-7""","""BOS""","""ORD""",867,9,18,0,null,0,0,0,0,0,0,64
2005,1,31,1,"""1556""",1605,1726,1759,"""UA""",541,"""N326UA""","""150""",174,"""129""","""-33""","""-9""","""BOS""","""ORD""",867,11,10,0,null,0,0,0,0,0,0,33
2005,1,2,7,"""1934""",1900,2235,2232,"""UA""",542,"""N902UA""","""121""",152,"""106""","""3""","""34""","""ORD""","""BOS""",867,5,10,0,null,0,0,0,0,0,0,3


We can verify that ArrTime is now numeric and AbsDiffArrTime was created correctly.

## Pledge

By submitting this work I hereby pledge that this is my own, personal work. I've acknowledged in the designated place at the top of this file all sources that I used to complete said work, including but not limited to: online resources, books, and electronic communications. I've noted all collaboration with fellow students and/or TA's. I did not copy or plagiarize another's work.

> As a Boilermaker pursuing academic excellence, I pledge to be honest and true in all that I do. Accountable together – We are Purdue.

https://www.purdue.edu/odos/osrr/honor-pledge/
